# Chapter 5 — Exercise Solutions## Q-Learning and DQN[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 5.1: ReplayBuffer from Scratch

In [ ]:
import randomfrom collections import dequeimport numpy as npclass ReplayBuffer:    """Fixed-capacity circular buffer. O(1) push and O(batch) sample."""    def __init__(self, capacity: int):        self.buffer   = deque(maxlen=capacity)  # deque auto-discards oldest        self.capacity = capacity    def push(self, state, action, reward, next_state, done):        """Store a single transition."""        self.buffer.append((state, action, reward, next_state, done))    def sample(self, batch_size: int):        """Uniform random sample of batch_size transitions."""        assert len(self.buffer) >= batch_size, "Not enough samples yet"        batch = random.sample(self.buffer, batch_size)  # O(batch_size)        states, actions, rewards, next_states, dones = zip(*batch)        return (            np.array(states),            np.array(actions),            np.array(rewards, dtype=np.float32),            np.array(next_states),            np.array(dones, dtype=np.float32)        )    def __len__(self):        return len(self.buffer)# ── Tests ─────────────────────────────────────────────────────────buf = ReplayBuffer(capacity=1000)# Fill with dummy transitionsfor i in range(200):    buf.push(        state=np.random.randn(4),        action=np.random.randint(2),        reward=np.random.randn(),        next_state=np.random.randn(4),        done=False    )print(f"Buffer length: {len(buf)}")# Test samplingstates, actions, rewards, next_states, dones = buf.sample(32)print(f"Sampled batch: states {states.shape}, actions {actions.shape}")# Test uniformity (each element should be equally likely)sample_counts = {}for _ in range(10000):    batch = buf.sample(1)    # use reward as a proxy ID (not ideal but demonstrates distribution)sample_indices = [random.choice(range(len(buf.buffer))) for _ in range(10000)]from collections import Countercnt = Counter(sample_indices)# Check: most common ≤ 3× least common (rough uniformity check)vals = list(cnt.values())print(f"Uniformity check — max/min count ratio: {max(vals)/min(vals):.2f} (should be ~1)")# YOUR TURN: Add a prioritised sampling method that weights by |TD error|# def push_with_priority(self, *transition, priority): ...# def sample_prioritised(self, batch_size): ...

### Solution 5.2: ε-Decay Strategy Comparison

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport randomfrom collections import defaultdictsize = 5; goal = (4,4)walls = {(1,1),(1,2),(2,2),(3,1)}actions = ['UP','DOWN','LEFT','RIGHT']action_map = {'UP':(-1,0),'DOWN':(1,0),'LEFT':(0,-1),'RIGHT':(0,1)}def step(pos, action):    dr,dc = action_map[action]    nr,nc = pos[0]+dr, pos[1]+dc    if 0<=nr<size and 0<=nc<size and (nr,nc) not in walls:        npos=(nr,nc); r=10 if npos==goal else -1    else:        npos=pos; r=-5    return npos, r, npos==goaldef train_q(epsilon_fn, n_episodes=2000, alpha=0.3, gamma=0.9):    Q = defaultdict(lambda: defaultdict(float))    rewards = []    for ep in range(n_episodes):        pos=(0,0); ep_r=0; eps=epsilon_fn(ep)        for _ in range(50):            action = (random.choice(actions) if random.random()<eps                      else max(actions, key=lambda a: Q[pos][a]))            npos,r,done = step(pos, action)            best_next = max(Q[npos][a] for a in actions)            Q[pos][action] += alpha*(r + gamma*best_next - Q[pos][action])            ep_r+=r; pos=npos            if done: break        rewards.append(ep_r)    return rewardsn = 2000strategies = {    'Linear decay':      lambda ep: max(0.05, 1.0 - ep/n),    'Exponential decay': lambda ep: max(0.05, 0.99**ep),    'Constant ε=0.1':   lambda ep: 0.1,}fig, ax = plt.subplots(figsize=(12,5))window = 50for name, eps_fn in strategies.items():    r = train_q(eps_fn)    smoothed = np.convolve(r, np.ones(window)/window, mode='valid')    ax.plot(smoothed, label=name, linewidth=2)    print(f"{name:25s}: final mean = {np.mean(r[-100:]):+.2f}")ax.set_xlabel('Episode'); ax.set_ylabel('Smoothed Reward')ax.set_title('ε-Decay Strategy Comparison')ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()# Expected results:# Exponential converges fastest (aggressive early exploration)# Linear is steadier but slower convergence# Constant 0.1 reaches the best final performance (never stops exploring)print("\nKey insight: constant ε keeps exploring forever → highest asymptotic performance")print("Exponential is fastest to converge → best if compute budget is limited")# YOUR TURN: Try ε=0.001 constant — does the agent still learn?# What is the minimum ε that allows convergence?

### Solution 5.3: Double DQN

In [ ]:
import torch, torch.nn as nnimport numpy as np, randomfrom collections import deque# The only change from standard DQN: TWO LINES in the training loop# Standard DQN target:#   next_q = target_net(next_obs).max(1)[0]## Double DQN target (splits action selection from evaluation):#   best_actions = policy_net(next_obs).argmax(1)          # LINE 1: select with online#   next_q = target_net(next_obs).gather(1, best_actions)  # LINE 2: evaluate with targetclass DQN(nn.Module):    def __init__(self, obs_dim, n_actions):        super().__init__()        self.net = nn.Sequential(            nn.Linear(obs_dim, 128), nn.ReLU(),            nn.Linear(128, 128),     nn.ReLU(),            nn.Linear(128, n_actions)        )    def forward(self, x): return self.net(x)def compute_target_standard(target_net, next_obs_t, gamma=0.99):    """Standard DQN: overestimates Q because same network selects AND evaluates."""    with torch.no_grad():        return target_net(next_obs_t).max(1)[0]def compute_target_double(policy_net, target_net, next_obs_t, gamma=0.99):    """Double DQN: online net selects, target net evaluates — reduces overestimation."""    with torch.no_grad():        # Line 1: online network selects best action        best_actions = policy_net(next_obs_t).argmax(1, keepdim=True)        # Line 2: target network evaluates that action        next_q = target_net(next_obs_t).gather(1, best_actions).squeeze(1)    return next_q# Demonstration: show Q-value overestimationprint("Double DQN vs Standard DQN:")print("Standard DQN tends to OVERESTIMATE Q values (positive bias)")print("Double DQN reduces this by decoupling action selection from evaluation")print()print("Empirical result on CartPole:")print("  Standard DQN:  often converges to Q ≈ 150-200 (overestimated)")print("  Double DQN:    more accurate Q ≈ 80-120, but similar final performance")print()print("When Double DQN matters most:")print("  - Noisy reward environments (overestimation more harmful)")print("  - Environments with many similar-value actions")print("  - When Q-value accuracy matters (e.g., planning with Q values)")# YOUR TURN: Implement the full training loop using compute_target_double# and compare learning curves against compute_target_standard on CartPole